In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub

linkgish_indosum_path = kagglehub.dataset_download('linkgish/indosum')

print('Data source import complete.')


Using Colab cache for faster access to the 'indosum' dataset.
Data source import complete.
Data source import complete.


In [2]:
# Mount Google Drive (untuk Colab)
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at: /content/drive/MyDrive")
DRIVE_PATH = "/content/drive/MyDrive/mbart_models"
IS_COLAB = True


# Create directory if not exists
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"📁 Models will be saved to: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at: /content/drive/MyDrive
📁 Models will be saved to: /content/drive/MyDrive/mbart_models


In [3]:
!nvidia-smi

Mon Feb 16 12:51:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q evaluate rouge_score

In [5]:
from datasets import load_dataset

data_files = {
    "train": "/kaggle/input/indosum/indosum/train.01.jsonl",
    "validation": "/kaggle/input/indosum/indosum/dev.01.jsonl",
    "test": "/kaggle/input/indosum/indosum/test.01.jsonl",
}

dataset = load_dataset("json", data_files=data_files)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['category', 'gold_labels', 'id', 'paragraphs', 'source', 'source_url', 'summary'],
        num_rows: 14262
    })
    validation: Dataset({
        features: ['category', 'gold_labels', 'id', 'paragraphs', 'source', 'source_url', 'summary'],
        num_rows: 750
    })
    test: Dataset({
        features: ['category', 'gold_labels', 'id', 'paragraphs', 'source', 'source_url', 'summary'],
        num_rows: 3762
    })
})

In [6]:
!cat /kaggle/input/indosum/indosum/README.txt

Description

This folder contains Indonesian text summarization dataset,
roughly 19K tokenized news articles from (formerly) Shortir.com,
a news aggregator site. The dataset is split into 5 cross-validation
folds, where each fold consists of a training set, a development set,
and a test set. The filenames have the following format (where X is
a fold number 1-5):

* train.0X.jsonl - training set for fold X
* dev.0X.jsonl - development set for fold X
* test.0X.jsonl - test set for fold X

Each file is in JSONL format, where each line is a JSON object corresponding
to an article. This JSON object has several keys:

* id - unique identifier of the article
* paragraphs - a list of paragraphs of the original article
* summary - gold summary of the article, given as a list of sentences
* gold_labels - gold extractive labels of the sentences in the article
* category - category to which the article belongs (in Indonesian)
* source - source of the article
* source_url - url of the original arti

In [7]:
def flatten_text(example):
    # paragraphs: list of paragraphs, each paragraph is list of sentences, each sentence is list of words
    # Join all words into sentences, then join all sentences into one text
    paragraphs = example["paragraphs"]
    flattened_paragraphs = []
    for paragraph in paragraphs:
        sentences = [" ".join(sentence) for sentence in paragraph]
        flattened_paragraphs.append(" ".join(sentences))
    
    # Join all paragraphs
    input_text = " ".join(flattened_paragraphs)
    
    # summary: list of sentences, each sentence is list of words
    summary = example["summary"]
    target_text = " ".join([" ".join(sentence) for sentence in summary])
    
    return {
        "input_text": input_text,
        "target_text": target_text
    }

# Convert nested lists to strings
dataset = dataset.map(flatten_text)

# Filter out empty or invalid entries
dataset = dataset.filter(
    lambda x: isinstance(x["input_text"], str) and isinstance(x["target_text"], str) 
    and len(x["input_text"]) > 0 and len(x["target_text"]) > 0
)

print(f"✅ Dataset prepared!")
print(f"Train samples: {len(dataset['train'])}")
print(f"Sample input length: {len(dataset['train'][0]['input_text'])} chars")
print(f"Sample target length: {len(dataset['train'][0]['target_text'])} chars")

Map:   0%|          | 0/14262 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/3762 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14262 [00:00<?, ? examples/s]

Filter:   0%|          | 0/750 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3762 [00:00<?, ? examples/s]

✅ Dataset prepared!
Train samples: 14262
Sample input length: 1968 chars
Sample target length: 377 chars


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/mbart-large-50"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "id_ID"
forced_bos_token_id = tokenizer.lang_code_to_id["id_ID"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie

In [9]:
def preprocess(examples):
    # Tokenize inputs - NO padding here (dynamic padding is faster)
    inputs = examples["input_text"]
    model_inputs = tokenizer(
        inputs, 
        max_length=512, 
        truncation=True,
        padding=False  # Let DataCollator handle dynamic padding
    )

    # Tokenize targets
    targets = examples["target_text"]
    labels = tokenizer(
        targets, 
        max_length=128, 
        truncation=True,
        padding=False
    )
    
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Apply preprocessing with caching for faster re-runs
tokenized_dataset = dataset.map(
    preprocess, 
    batched=True,
    batch_size=1000,  # Larger batch for faster preprocessing
    remove_columns=dataset["train"].column_names,
    num_proc=4,  # Parallel processing
    desc="Tokenizing dataset"
)

print("✅ Preprocessing complete!")
print(f"Columns: {tokenized_dataset['train'].column_names}")
print(f"Train samples: {len(tokenized_dataset['train'])}")

Map:   0%|          | 0/14262 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/3762 [00:00<?, ? examples/s]

✅ Preprocessing complete!
Columns: ['input_ids', 'attention_mask', 'labels']


In [10]:
print(tokenized_dataset["train"].column_names)

['input_ids', 'attention_mask', 'labels']


In [11]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result


In [12]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch

# Clear CUDA cache first
torch.cuda.empty_cache()

# Check GPU memory
print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Use Drive path for saving checkpoints
output_dir = f"{DRIVE_PATH}/mbart_indosum_checkpoints"

# OPTIMIZED FOR A100 - Much more aggressive settings!
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    
    # LEARNING RATE - slightly higher for larger batches
    learning_rate=3e-5,
    
    # BATCH SIZE - A100 can handle MUCH larger batches!
    per_device_train_batch_size=8,              # 1 → 8 (8x more!)
    per_device_eval_batch_size=16,              # 1 → 16 (faster eval)
    gradient_accumulation_steps=2,              # 4 → 2 (effective batch = 16)
    
    # OPTIMIZATION
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    
    # EPOCHS & CHECKPOINTING
    num_train_epochs=3,
    save_total_limit=2,                         # Save disk space
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    
    # PRECISION - A100 supports BF16 (better than FP16!)
    bf16=True,                                  # Use BF16 instead of FP16 on A100
    bf16_full_eval=True,
    tf32=True,                                  # Enable TF32 for even faster matmul
    
    # GENERATION
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    
    # LOGGING - Anda akan melihat semua progress di sini!
    logging_dir=f"{DRIVE_PATH}/logs",
    logging_steps=50,                           # Log setiap 50 steps (lebih sering)
    logging_first_step=True,                    # Log step pertama
    report_to=["tensorboard"],                  # TensorBoard logging
    
    # DATALOADER - A100 has fast memory bus, use it!
    dataloader_num_workers=4,                   # Parallel data loading
    dataloader_pin_memory=True,                 # Faster GPU transfer
    dataloader_prefetch_factor=2,               # Prefetch batches
    
    # OPTIMIZATION FLAGS
    gradient_checkpointing=False,               # DISABLE - A100 has enough memory
    optim="adamw_torch_fused",                  # Fused AdamW (faster on A100)
    warmup_ratio=0.05,                          # 5% warmup
    lr_scheduler_type="cosine",                 # Cosine decay
    
    # MISC
    push_to_hub=False,
    disable_tqdm=False,                         # ENABLE progress bar!
    remove_unused_columns=True,
    eval_accumulation_steps=8,                  # Save memory during eval
)

# DataCollator with dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer, 
    model=model,
    padding=True,  # Dynamic padding
    max_length=512
)

# Move model to GPU
model = model.to("cuda")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Calculate expected training time
samples_per_step = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = len(tokenized_dataset["train"]) // samples_per_step
total_steps = steps_per_epoch * training_args.num_train_epochs

print("\n" + "="*60)
print("✅ TRAINER INITIALIZED (OPTIMIZED FOR A100)!")
print("="*60)
print(f"📊 Effective batch size: {samples_per_step}")
print(f"📊 Steps per epoch: {steps_per_epoch}")
print(f"📊 Total optimizer steps: {total_steps}")
print(f"📊 Training samples: {len(tokenized_dataset['train']):,}")
print(f"💾 Checkpoints: {output_dir}")
print(f"⚡ Expected speed: ~60-100 samples/sec on A100")
print(f"⏱️  Estimated time per epoch: ~{len(tokenized_dataset['train']) / 75 / 60:.1f} minutes")
print(f"⏱️  Total training time: ~{3 * len(tokenized_dataset['train']) / 75 / 60:.1f} minutes (3 epochs)")
print("="*60)
print("\n📊 LOG YANG AKAN DITAMPILKAN:")
print("  ✓ Progress bar per batch (disable_tqdm=False)")
print("  ✓ Loss updates setiap 50 steps (logging_steps=50)")
print("  ✓ Evaluasi ROUGE di akhir setiap epoch")
print("  ✓ Checkpoint saves dengan konfirmasi")
print("  ✓ Estimasi waktu remaining (ETA)")
print("="*60)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer initialized!
📊 Epochs: 3
💾 Checkpoints saved to: /content/drive/MyDrive/mbart_models/mbart_indosum_checkpoints
📁 Accessible in Google Drive: MyDrive/mbart_models/
🏆 Best model will be loaded at the end


In [ ]:
# Resume training from checkpoint (jika terputus)
import os
import glob

checkpoint_dir = f"{DRIVE_PATH}/mbart_indosum_checkpoints"

# Cari checkpoint terakhir
checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
if checkpoints:
    # Sort by step number
    checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))
    latest_checkpoint = checkpoints[-1]
    print(f"🔄 Found checkpoint: {latest_checkpoint}")
    print(f"▶️  Resuming training from checkpoint...")
    
    # Resume training
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print(f"❌ No checkpoint found in {checkpoint_dir}")
    print(f"▶️  Starting fresh training...")
    
    # Start fresh training
    trainer.train()

In [ ]:
trainer.evaluate(tokenized_dataset["test"])


In [ ]:
# Save the fine-tuned model to Google Drive
output_model_dir = f"{DRIVE_PATH}/mbart_indosum_final"

print(f"💾 Saving model to {output_model_dir}...")
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print("✅ Model saved successfully!")
print(f"\n📂 Model location: {output_model_dir}")
if IS_COLAB:
    print(f"🔗 Google Drive: MyDrive/mbart_models/mbart_indosum_final/")
print("\n🔄 To load the model later:")
print(f'  from transformers import AutoTokenizer, AutoModelForSeq2SeqLM')
print(f'  tokenizer = AutoTokenizer.from_pretrained("{output_model_dir}")')
print(f'  model = AutoModelForSeq2SeqLM.from_pretrained("{output_model_dir}")')

In [ ]:
# Upload model to Hugging Face Hub
from huggingface_hub import notebook_login

# Optional: Set git credential helper to suppress warning
import os
os.system("git config --global credential.helper store")

# Login to Hugging Face (will prompt for token)
notebook_login()

# Set your model name on HF Hub
hf_model_name = "your-username/mbart-indonesian-summarization"  # Change this!

print(f"🚀 Uploading model to: {hf_model_name}")

# Push model and tokenizer
model.push_to_hub(hf_model_name, private=False)
tokenizer.push_to_hub(hf_model_name, private=False)

print(f"✅ Model uploaded successfully!")
print(f"🔗 Model URL: https://huggingface.co/{hf_model_name}")